# PS LiDAR - Development Playground

**Available Bricks:**
- Brick 1: Data Loading
- Brick 2: Circular Clipping (manual coordinates)
- Brick 3: Normalisation Analysis
- Brick 4: Ground Filtering
- Brick 5: Height Normalisation + Export Checkpoint
- Brick 6: 3D Visualisation
- Brick 7: **Trunk Extraction** (verticality + DBSCAN + PCA)
- Brick 8: **Branch Extraction** (linearity + connectivity)
- Brick 9: **Export & Visualisation**


In [1]:
import os
import sys
import time
import numpy as np
from pathlib import Path

# Resolve project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.core import (
    PointCloudLoader,
    clip_circular_plot,
    detect_normalization,
    classify_ground,
    normalize_heights,
    export_point_cloud,
)

print(f"Project root: {project_root}")
print("All imports loaded successfully.")


Project root: c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR
All imports loaded successfully.


In [2]:
FILE_PATH = "D:/LiDAR QAs/T460298_subsampled_laz1_4.laz"

loader = PointCloudLoader(FILE_PATH)
loader.load()

meta = loader.get_metadata()
print(f"File: {meta['filename']}")
print(f"Points: {meta['point_count']:,}")
print(f"Size: {meta['file_size_mb']} MB")

File: T460298_subsampled_laz1_4.laz
Points: 38,249,740
Size: 449.65 MB


In [3]:
# Load XYZ and available scalar fields
xyz_full = loader.get_xyz()

scalar_fields = {}
for field in ['intensity', 'return_number', 'number_of_returns', 'classification']:
    try:
        scalar_fields[field] = loader.get_attribute(field)
        print(f" {field}: {len(scalar_fields[field]):,} values")
    except:
        print(f" {field}: not available")

print(f"\nXYZ memory: {xyz_full.nbytes / (1024**2):.1f} MB")

 intensity: 38,249,740 values
 return_number: 38,249,740 values
 number_of_returns: 38,249,740 values
 classification: 38,249,740 values

XYZ memory: 437.7 MB


---
## 2. Circular Clipping (Brick 2)

**Instructions:**
1. Open the original file in CloudCompare
2. Use the "Point Picking" tool to locate the mat centre
3. Copy the Xg, Yg coordinates shown
4. Paste the values into `CENTER_X` and `CENTER_Y` below

In [4]:
# 
# USER PARAMETERS - Modify per plot
# 

# Centre coordinates (from CloudCompare Point Picking)
CENTER_X = -0.311372
CENTER_Y = -1.461118

# Plot radius in metres
PLOT_RADIUS = 16.0

# 

print(f"Centre: ({CENTER_X:.6f}, {CENTER_Y:.6f})")
print(f"Radius: {PLOT_RADIUS}m")

Centro: (-0.311372, -1.461118)
Radio: 16.0m


In [5]:
# Run circular clipping
t0 = time.perf_counter()
clip_result = clip_circular_plot(xyz_full, CENTER_X, CENTER_Y, PLOT_RADIUS)
elapsed = time.perf_counter() - t0

plot_indices = clip_result.indices

print(f" Clipped in {elapsed*1000:.0f}ms")
print(f"Original points: {len(xyz_full):,}")
print(f"Points in plot: {clip_result.n_points:,} ({clip_result.n_points/len(xyz_full):.1%})")

 Clipped in 712ms
Original points: 38,249,740
Points in plot: 21,502,878 (56.2%)


In [6]:
# Apply clipping to XYZ and scalar fields
xyz = xyz_full[plot_indices]

plot_scalars = {}
for field, values in scalar_fields.items():
    plot_scalars[field] = values[plot_indices]

print(f"Plot XYZ: {xyz.shape}")
print(f"Scalar fields: {list(plot_scalars.keys())}")

# Free memory
del xyz_full, scalar_fields
import gc; gc.collect()
print(" Memory freed")

Plot XYZ: (21502878, 3)
Scalar fields: ['intensity', 'return_number', 'number_of_returns', 'classification']
 Memory freed


---
## 3. Normalisation Analysis (Brick 3)

In [7]:
analysis = detect_normalization(xyz)
print(f"Status: {analysis.status.value.upper()}")
print(f"Normalised: {analysis.is_normalized}")
print(f"Z range: {analysis.z_min:.2f}m to {analysis.z_max:.2f}m")

Estatus: NOT_NORMALIZED
Normalised?: False
Rango Z: -3.91m a 32.66m


---
## 4. Ground Filtering (Brick 4)

In [10]:
from src.core.ground import classify_ground
print("Running CSF...")
t0 = time.perf_counter()

ground_result = classify_ground(
    xyz,
    cloth_resolution=1.0,
    rigidness=1,
    class_threshold=0.5,
    slope_smooth=True,
)

print(f" Completed in {time.perf_counter() - t0:.2f}s")
print(f"Ground: {ground_result.n_ground:,} ({ground_result.ground_ratio:.1%})")
print(f"Vegetation: {ground_result.n_off_ground:,}")

# Extract ground and vegetation point clouds
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

print(f"Ground points: {len(ground_xyz):,}")
print(f"Vegetation points: {len(vegetation_xyz):,}")


Running CSF...
 Completed in 11.53s
Ground: 3,988,415 (18.5%)
Vegetation: 17,514,463
Ground points: 3,988,415
Vegetation points: 17,514,463


In [11]:
# Separate ground and vegetation
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

ground_scalars = {k: v[ground_result.ground_indices] for k, v in plot_scalars.items()}
vegetation_scalars = {k: v[ground_result.off_ground_indices] for k, v in plot_scalars.items()}

print(f"Ground: {len(ground_xyz):,} points")
print(f"Vegetation: {len(vegetation_xyz):,} points")

Ground: 3,988,415 points
Vegetation: 17,514,463 points


---
## 5. Height Normalisation (Brick 5)

In [12]:
print("Normalising heights...")
t0 = time.perf_counter()

veg_norm_result = normalize_heights(vegetation_xyz, ground_xyz, resolution=0.5)
veg_normalized = veg_norm_result.xyz_normalized

ground_norm_result = normalize_heights(ground_xyz, ground_xyz, resolution=0.5)
ground_normalized = ground_norm_result.xyz_normalized

print(f" Completed in {(time.perf_counter() - t0)*1000:.0f}ms")
print(f"")
print(f"Vegetation: Z = {veg_normalized[:, 2].min():.2f}m to {veg_normalized[:, 2].max():.2f}m")
print(f"Ground: Z = {ground_normalized[:, 2].min():.2f}m to {ground_normalized[:, 2].max():.2f}m")

Normalising heights...
 Completed in 2899ms

Vegetation: Z = -1.23m a 34.44m
Ground: Z = -0.90m a 0.82m


---
## 5.5 Export Checkpoint

In [15]:
from pathlib import Path
# Output directory
OUTPUT_DIR = Path("D:/OUTPUTS/T460298A")

# Export vegetation
veg_file = Path("D:/OUTPUTS/T460298A_VEG_NORM.laz")
export_point_cloud(
    veg_file,
    veg_normalized,
    intensity=vegetation_scalars.get('intensity'),
    return_number=vegetation_scalars.get('return_number'),
    number_of_returns=vegetation_scalars.get('number_of_returns'),
    classification=vegetation_scalars.get('classification'),
)
print(f"✓ Vegetation: {veg_file.name} ({veg_file.stat().st_size / (1024**2):.1f} MB)")

# Export ground
ground_file = Path("D:/OUTPUTS/T460298A_GROUND_NORM.laz")
export_point_cloud(
    ground_file,
    ground_normalized,
    intensity=ground_scalars.get('intensity'),
    return_number=ground_scalars.get('return_number'),
    number_of_returns=ground_scalars.get('number_of_returns'),
    classification=ground_scalars.get('classification'),
)
print(f" Ground: {ground_file.name} ({ground_file.stat().st_size / (1024**2):.1f} MB)")

NameError: name 'Path' is not defined

---
## 6. 3D Visualisation (Brick 6)

In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(veg_normalized)

# Colour by height
z = veg_normalized[:, 2]
z_scaled = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((len(z_scaled), 3))
colors[:, 0] = z_scaled
colors[:, 1] = 1 - np.abs(2 * z_scaled - 1)
colors[:, 2] = 1 - z_scaled
pcd.colors = o3d.utility.Vector3dVector(colors)

print(f"Cloud: {len(pcd.points):,} points")

In [ ]:
o3d.visualization.draw_geometries([pcd], window_name="Normalised Vegetation", width=1280, height=720)


---
---
# BRICK 7: Trunk Extraction

**Pipeline** (dendromatics/3DFin):
1. Extract horizontal stripe at breast height
2. Voxelise and compute verticality (pgeof C++)
3. DBSCAN clustering in 2D (XY)
4. Iterative peeling
5. PCA for tree axes
6. Assign all points to the nearest axis

**Field parameters** (measured before/after scanning):


In [ ]:
# ==============================================================
# BRICK 7: TRUNK EXTRACTION
# ==============================================================

import os, sys, time
import numpy as np
import laspy
from pathlib import Path

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.core.trunk_extraction import extract_trunks, TrunkExtractionConfig

# ---- FIELD-MEASURED PARAMETERS (modify per plot) ----

# Tree dimensions
DBH_MIN = 0.20          # metres — smallest expected stem diameter
DBH_MAX = 0.80          # metres — largest expected stem diameter
HEIGHT_MAX = 36.0       # metres — tallest tree in the plot

# Breast-height detection band
STRIPE_LOWER = 2.0      # metres — lower limit of detection stripe
STRIPE_UPPER = 6.0      # metres — upper limit of detection stripe

# Crown distances (horizontal distance from trunk to tip of longest
# branch, measured per cardinal direction on the central tree)
CROWN_DISTANCE_A = 8.0  # metres — North
CROWN_DISTANCE_B = 7.0  # metres — East
CROWN_DISTANCE_C = 6.5  # metres — South
CROWN_DISTANCE_D = 7.5  # metres — West

# Derived: max crown radius = axis assignment distance
MAX_AXIS_DISTANCE = max(CROWN_DISTANCE_A, CROWN_DISTANCE_B,
                        CROWN_DISTANCE_C, CROWN_DISTANCE_D)
# Derived: stem search radius (half the max DBH, with a small margin)
STEM_SEARCH_RADIUS = (DBH_MAX / 2) + 0.10

print(f"Crown distances: A={CROWN_DISTANCE_A}m, B={CROWN_DISTANCE_B}m, "
      f"C={CROWN_DISTANCE_C}m, D={CROWN_DISTANCE_D}m")
print(f"Max axis distance (derived): {MAX_AXIS_DISTANCE}m")
print(f"Stem search radius (derived): {STEM_SEARCH_RADIUS}m")

# ---- ALGORITHM PARAMETERS ----

VOXEL_RESOLUTION = 0.05   # metres
VERTICALITY_THRESH = 0.7  # 0-1 (higher = stricter)
PEELING_ITERATIONS = 2    # passes of verticality peeling
MIN_CLUSTER_PTS = 500     # minimum voxels per stem cluster

# ---- LOAD CHECKPOINT ----

VEG_CHECKPOINT = Path("D:/OUTPUTS/T460298A_VEG_NORM.laz")

print(f"\nLoading: {VEG_CHECKPOINT.name}")
t0 = time.perf_counter()
las = laspy.read(str(VEG_CHECKPOINT))
veg_normalized = np.column_stack([
    np.array(las.x, dtype=np.float32),
    np.array(las.y, dtype=np.float32),
    np.array(las.z, dtype=np.float32),
])
print(f"  {len(veg_normalized):,} points in {time.perf_counter()-t0:.1f}s")
print(f"  Z range: {veg_normalized[:,2].min():.2f}m to {veg_normalized[:,2].max():.2f}m")

# ---- RUN EXTRACTION ----

trunk_config = TrunkExtractionConfig(
    stripe_lower_limit=STRIPE_LOWER,
    stripe_upper_limit=STRIPE_UPPER,
    dbh_min=DBH_MIN,
    dbh_max=DBH_MAX,
    height_max=HEIGHT_MAX,
    max_axis_distance=MAX_AXIS_DISTANCE,
    stem_search_radius=STEM_SEARCH_RADIUS,
    voxel_resolution_xy=VOXEL_RESOLUTION,
    voxel_resolution_z=VOXEL_RESOLUTION,
    verticality_threshold=VERTICALITY_THRESH,
    peeling_iterations=PEELING_ITERATIONS,
    min_cluster_points=MIN_CLUSTER_PTS,
)

print(f"\nInput points: {len(veg_normalized):,}")
trunk_result = extract_trunks(veg_normalized, trunk_config, verbose=True)
print(f"\nTrees found: {trunk_result.n_trees}")


In [ ]:
# ==============================================================
# BRICK 7 (cont): SEPARATE TRUNK vs NON-TRUNK
# ==============================================================

# trunk_mask = points within stem_search_radius of an axis
# tree_ids = which tree each point belongs to (-1 = unassigned)

trunk_points = veg_normalized[trunk_result.trunk_mask]
non_trunk_mask = ~trunk_result.trunk_mask
non_trunk_points = veg_normalized[non_trunk_mask]

print(f"Trunk points:     {len(trunk_points):,} ({trunk_result.trunk_mask.mean():.1%})")
print(f"Non-trunk points: {len(non_trunk_points):,}")
print(f"Assigned to a tree: {(trunk_result.tree_ids >= 0).sum():,}")
print(f"Unassigned:         {(trunk_result.tree_ids == -1).sum():,}")


---
# BRICK 8: Branch Extraction

**Pipeline:**
1. Compute linearity (pgeof) on non-trunk points
2. Filter by linearity threshold
3. 26-neighbour connectivity graph
4. Keep only components connected to trunks
5. Filter by maximum branch length


In [ ]:
# ==============================================================
# BRICK 8: BRANCH EXTRACTION
# ==============================================================

from src.core.branch_extraction import extract_branches, BranchExtractionConfig

# ---- FIELD-MEASURED PARAMETERS ----
MAX_BRANCH_LENGTH = 8.0     # metres — longest expected branch

# ---- ALGORITHM PARAMETERS ----
LINEARITY_THRESH = 0.5      # 0-1 (higher = stricter)
CONNECTIVITY_RADIUS = 0.05  # metres (voxel size for connectivity graph)
MIN_BRANCH_POINTS = 50      # minimum points per branch cluster

branch_config = BranchExtractionConfig(
    max_branch_length=MAX_BRANCH_LENGTH,
    linearity_threshold=LINEARITY_THRESH,
    connectivity_radius=CONNECTIVITY_RADIUS,
    min_branch_points=MIN_BRANCH_POINTS,
)

print(f"Branch config: {branch_config}")
print(f"Input points: {len(veg_normalized):,}")
print(f"Trunk points: {trunk_result.trunk_mask.sum():,}")

branch_result = extract_branches(
    veg_normalized,
    trunk_result,
    branch_config,
    verbose=True
)

print(f"\nBranch points: {branch_result.n_branch_points:,}")
print(f"Wood (trunk+branch): {branch_result.wood_mask.sum():,} "
      f"({branch_result.wood_mask.mean():.1%})")


---
# BRICK 9: Export & Visualisation

Exports separated clouds for validation in CloudCompare.


In [ ]:
# ==============================================================
# BRICK 9: EXPORT & TREE INVENTORY
# ==============================================================

from src.core.io import export_point_cloud
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path("D:/OUTPUTS/T460298A")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Build tree_id classification for the entire wood cloud ---
# tree_ids: per-point tree assignment from Brick 7
# Clamp to uint8 range (0-254, 255 = unassigned)
tree_ids_clamped = trunk_result.tree_ids.copy()
tree_ids_clamped[tree_ids_clamped == -1] = 255
tree_ids_clamped = tree_ids_clamped.astype(np.uint8)

# --- Export LAZ files with tree IDs ---

# Trunks (with tree ID)
trunk_pts = veg_normalized[trunk_result.trunk_mask]
trunk_ids = tree_ids_clamped[trunk_result.trunk_mask]
export_point_cloud(OUTPUT_DIR / "trunks.laz", trunk_pts, classification=trunk_ids, point_format=6)
print(f"✓ Trunks: {len(trunk_pts):,} points")

# Branches (with tree ID)
branch_pts = veg_normalized[branch_result.branch_only_mask]
branch_ids = tree_ids_clamped[branch_result.branch_only_mask]
export_point_cloud(OUTPUT_DIR / "branches.laz", branch_pts, classification=branch_ids, point_format=6)
print(f"✓ Branches: {len(branch_pts):,} points")

# Wood structure (trunk + branches, with tree ID)
wood_pts = veg_normalized[branch_result.wood_mask]
wood_ids = tree_ids_clamped[branch_result.wood_mask]
export_point_cloud(OUTPUT_DIR / "wood_structure.laz", wood_pts, classification=wood_ids, point_format=6)
print(f"✓ Wood structure: {len(wood_pts):,} points")

# --- Tree Inventory (Excel) ---

rows = []
for ax in trunk_result.tree_axes:
    tid = ax['tree_id']
    cx, cy = ax['centroid'][0], ax['centroid'][1]
    n_pts = (trunk_result.tree_ids == tid).sum()
    rows.append({
        'Tree_ID': tid,
        'X': round(float(cx), 3),
        'Y': round(float(cy), 3),
        'Z_min': round(float(ax['z_min']), 2),
        'Z_max': round(float(ax['z_max']), 2),
        'N_points': int(n_pts),
    })

df = pd.DataFrame(rows)
xlsx_path = OUTPUT_DIR / "tree_inventory.xlsx"
df.to_excel(xlsx_path, index=False, sheet_name='Tree Inventory')

print(f"\n✓ Tree inventory: {xlsx_path.name} ({len(df)} trees)")
print(f"\nAll exports saved to {OUTPUT_DIR}")
df


### Visualisation (Open3D)


In [ ]:
# ==============================================================
# BRICK 9 (cont): 3D VISUALISATION
# ==============================================================

import open3d as o3d
import numpy as np

# Colour trunks brown, branches green
trunk_colours = np.tile([0.6, 0.4, 0.2], (len(veg_normalized[trunk_result.trunk_mask]), 1))
branch_colours = np.tile([0.2, 0.7, 0.3], (len(branch_result.branch_points), 1))

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(wood)
pcd.colors = o3d.utility.Vector3dVector(np.vstack([trunk_colours, branch_colours]))

o3d.visualization.draw_geometries([pcd], window_name="Wood Structure")


---
## Next Steps

- **Brick 10:** Per-tree analysis (DBH, height, sweep)
- **Brick 11:** Fork detection
- **Brick 12:** HQP classification of branches and spikes
